### Now the main step is to load the data from Staging Table to Target Table using merge commands

In [0]:
from delta.tables import *

stage_table_name = "dev_ext_loc.default.stg_table"
target_table_name = "dev_ext_loc.default.tgt_table"

- **Iteration 1** : Initally you will load all the data from Staging Table to Target Table
- **Iteration_1_onwards** : You will compare the data between these 2 tables and load the necessary data to the Target Table 

In [0]:
stage_df = sprak.read.table(stage_table_name)

- **Now, create a Target Table from the Staging Table if Target Table does not exist**
- **if it exists then perform merge**

In [0]:
if not spark._jsparkSession.catalog().tableExists(target_table_name):
    stage_df.write.format("delta").saveAsTable(target_table_name)

else:
    target_table = DeltaTable.forName(spark, target_table_name)

    # merge condition
    merge_condition = "stage_df.tracking_num = target_table.tracking_num"  

    # merge query logic
    target_table.merge(stage_df, merge_condition)\
                                                .whenMatchedDelete()\
                                                .execute()
    # the logic says that if the entry inside the stage matches the target, meaning a fresh record with udated column values for the same row has arrived. Therefore delete the entry from the target and then append the newly record of stage table in target table
    
    stage_df.write.format("delta").mode("append").saveAsTable(target_table_name)
